<a href="https://colab.research.google.com/github/Nitish-9k/ATS_System/blob/main/JupyterNotebook/FineTuningBert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util
from sentence_transformers.evaluation import  EmbeddingSimilarityEvaluator
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
import json , warnings
warnings.filterwarnings("ignore")

print("libraries imported")

libraries imported


In [ ]:
df=pd.read_csv("cleaned_resumeJD_pairs (2).csv")
print(f"loaded:{len(df)} pairs")
print(" label distribution :")
print(df["match_label"].value_counts())
df.head(3)


loaded:266 pairs
 label distribution :
match_label
high      107
low        81
medium     78
Name: count, dtype: int64


,resume_text,job_description,match_score,match_label,match_lower_label,resume_len,jd_len
0,Operations Analyst with 12 years of experience...,Network Engineer at nonprofit organization. Re...,0.16,low,low,36,33
1,Cybersecurity Analyst with 7 years of experien...,Financial Analyst at financial institution. Re...,0.08,low,low,38,33
2,Supply Chain Analyst with 8 years of experienc...,Registered Nurse at financial institution. Req...,0.91,high,high,52,48


In [ ]:
train_df,temp_df=train_test_split(df,test_size=0.3,random_state=42,stratify=df["match_label"])

val_df,test_df=train_test_split(temp_df,test_size=0.5,random_state=42,stratify=temp_df["match_label"])


print(f"train_df:{len(train_df)}")
print(f"val_df:{len(val_df)}")
print(f"test_df:{len(test_df)}")

# verify each label has 3 labels

for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"\n{name}:")
    print(df["match_label"].value_counts())


train_df:186
val_df:40
test_df:40

train:
match_label
high      75
low       57
medium    54
Name: count, dtype: int64

validation:
match_label
high      16
low       12
medium    12
Name: count, dtype: int64

test:
match_label
high      16
low       12
medium    12
Name: count, dtype: int64


In [ ]:
# convert  to inputExample format- senternce transformer
#
from sentence_transformers import InputExample

train_examples=[
    InputExample(texts=[row["resume_text"],row["job_description"]],label=float(row["match_score"]))


    for _,row in train_df.iterrows()
]
val_examples=[InputExample(texts=[row["resume_text"],row["job_description"]],label=float(row["match_score"]))




    for _, row in val_df.iterrows()

]


print(f"Train Examples : {len(train_examples)}")
print(f"Train Examples : {len(val_examples)}")
print("Samples")
print(f"train_examples{train_examples[0]}")
print(f"val_examples{val_examples[0]}")


Train Examples : 186
Train Examples : 40
Samples
train_examples<InputExample> label: 0.95, texts: UX Designer with 6 years of experience specializing in user research, wireframing, and prototyping. Delivered measurable results through cross-functional collaboration and data-driven decision making. Proficient in industry-standard tools and passionate about continuous improvement. (Profile ref #104) Consistently exceeded targets set for the role. Collaborated closely with senior leadership on strategic goals.; Graphic Designer at SaaS company. Required skills: branding, layout design, and Adobe Creative Suite. Looking for a motivated professional to join a growing team and contribute to key initiatives. Competitive compensation and benefits. (Req #448) Recognized for strong performance and reliability. Collaborated closely with senior leadership on strategic goals.
val_examples<InputExample> label: 0.91, texts: QA Engineer with 3 years of experience specializing in test automation, regre

In [ ]:

from transformers.data.processors.utils import InputExample

In [ ]:

print("loading base bert model")
base_model=SentenceTransformer("all-mpnet-base-v2")
print("model loaded")

loading base bert model


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model loaded


In [ ]:
print("Evaluating the Base Model on test set")

base_preds = []

for _, row in test_df.iterrows():
    emb1 = base_model.encode(row["resume_text"])
    emb2 = base_model.encode(row["job_description"])

    sim = cosine_similarity([emb1], [emb2])[0][0]
    base_preds.append(float(sim))

# Actual scores
y_true = test_df["match_score"].values

# MAE
base_mae = mean_absolute_error(y_true, base_preds)

# RMSE
base_rmse = np.sqrt(mean_squared_error(y_true, base_preds))

print(f"Base Model MAE  : {base_mae:.4f}")
print(f"Base Model RMSE : {base_rmse:.4f}")

Evaluating the Base Model on test set
Base Model MAE  : 0.3019
Base Model RMSE : 0.3524


In [ ]:
from sentence_transformers import losses
model=SentenceTransformer("all-mpnet-base-v2")
train_dataloader=DataLoader(train_examples,shuffle=True,batch_size=16)
train_loss=losses.CosineSimilarityLoss(model)

evaluator=EmbeddingSimilarityEvaluator.from_input_examples(
    val_examples,
    name="ats-validation"
)

print("training setup ready ")
print("batch size : 16")
print(f"Train pairs :{len(train_examples)}")
print(f"steps/epochs:{len(train_dataloader)}")




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

training setup ready 
batch size : 16
Train pairs :186
steps/epochs:12


In [ ]:
total_Steps=len(train_dataloader)*10
warmup_steps=total_Steps*0.1
epochs=10

print(f"total_Steps:{total_Steps}")
print(f"warmup_steps:{warmup_steps}")

model.fit(
    train_objectives=[(train_dataloader,train_loss)],
    evaluator=evaluator,
    evaluation_steps=len(train_dataloader),
    warmup_steps=warmup_steps,
    output_path="./ats_model",
    save_best_model=True,
    use_amp=True,
    epochs=epochs,
    # save_steps=total_Steps,
    show_progress_bar=True


)

print("/n Fine tuning Complete ")



total_Steps:120
warmup_steps:12.0


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Ats-validation Pearson Cosine,Ats-validation Spearman Cosine
12,No log,No log,0.888504,0.770388
24,No log,No log,0.968920,0.882188
36,No log,No log,0.972963,0.877585
48,No log,No log,0.981455,0.891207
60,No log,No log,0.984560,0.882470
72,No log,No log,0.984236,0.885758
84,No log,No log,0.987052,0.881155
96,No log,No log,0.986289,0.882188
108,No log,No log,0.987656,0.879370
120,No log,No log,0.987653,0.879370


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/n Fine tuning Complete 


In [ ]:
fine_tuned_model=SentenceTransformer("ats_model")
print("fine tuned model loaded ")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

fine tuned model loaded 


In [ ]:
print("Evaluating the Base Model on test set")

ft_preds= []

for _, row in test_df.iterrows():
    emb1 = fine_tuned_model.encode(row["resume_text"])
    emb2 = fine_tuned_model.encode(row["job_description"])

    sim = cosine_similarity([emb1], [emb2])[0][0]
    ft_preds.append(float(sim))

# Actual scores
y_true = test_df["match_score"].values

# MAE
ft_mae = mean_absolute_error(y_true, ft_preds)

# RMSE
ft_rmse = np.sqrt(mean_squared_error(y_true, ft_preds))

print(f"Fine_Tuned_Model MAE  : {ft_mae:.4f}")
print(f"Fine_Model RMSE : {ft_rmse:.4f}")

Evaluating the Base Model on test set
Fine_Tuned_Model MAE  : 0.0574
Fine_Model RMSE : 0.0714


In [ ]:
metadata={
    "base_model":str(base_model),
    "dataset": "merged_dataset_clean.csv",

    "train_df":len(train_df),
    "val_df":len(val_df),
    "test_df":len(test_df),
    "epochs":epochs,
    "batch_size":16,
    "base_mae":float(base_mae),
    "base_rmse":float(base_rmse),
    "ft_mae":float(ft_mae),
    "ft_rmse":float(ft_rmse),
}

with open("models/finetuned-bert/metadata.json","w") as f:
    json.dump(metadata,f,indent=4)


print("metadata saved")


{'base_model': "SentenceTransformer(\n  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})\n  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})\n  (2): Normalize({})\n)", 'dataset': 'merged_dataset_clean.csv', 'train_df': 186, 'val_df': 40, 'test_df': 40, 'epochs': 10, 'batch_size': 16, 'base_mae': 0.30190324631333354, 'base_rmse': 0.3524096957781745, 'ft_mae': 0.05735976609960199, 'ft_rmse': 0.07143676841196742}
metadata saved


In [ ]:
print(json.dumps(metadata,indent=4))

{
    "base_model": "SentenceTransformer(\n  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})\n  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})\n  (2): Normalize({})\n)",
    "dataset": "merged_dataset_clean.csv",
    "train_df": 186,
    "val_df": 40,
    "test_df": 40,
    "epochs": 10,
    "batch_size": 16,
    "base_mae": 0.30190324631333354,
    "base_rmse": 0.3524096957781745,
    "ft_mae": 0.05735976609960199,
    "ft_rmse": 0.07143676841196742
}


In [ ]:
import shutil

shutil.make_archive("ats_model", "zip", "./ats_model")

print("Model zipped successfully!")


Model zipped successfully!


In [ ]:
from google.colab import files
files.download("ats_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>